In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder,OrdinalEncoder,PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet, SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import clone
from xgboost import XGBRegressor


ModuleNotFoundError: No module named 'xgboost'

In [ ]:
import sys
print(sys.executable)

In [ ]:
import kagglehub
import os

# Download / load from Colab cache
path = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")

# Check the exact filename(s) inside the folder
files = os.listdir(path)
print("Files found:", files)

# Load the CSV file (usually 'realtor-data.zip.csv')
df = pd.read_csv(os.path.join(path, files[0]))

df.head()


In [ ]:
df.shape

In [ ]:

df.head()

In [ ]:
df.info()

In [ ]:
print("Null values: \n",df.isnull().sum())
print("Duplicate instances: ", df.duplicated().sum())

In [ ]:
df.dropna(subset=["price"], inplace= True)
df["price"].isnull().sum()

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": df.isnull().mean() * 100
})

print(missing.sort_values("Missing %", ascending=False))

In [ ]:
numerical_cols = df.select_dtypes(include="number")
for col in numerical_cols:
    print(col,":\n\t",df[df[col]<0],"\n\n")

In [ ]:
df["prev_sold_date"] = pd.to_datetime(
    df["prev_sold_date"],
    errors="coerce"
)

reference_date = pd.Timestamp("2026-11-11")

df["years_since_sold"] = (
    (reference_date - df["prev_sold_date"]).dt.days / 365.25
)

df["sold_year"] = df["prev_sold_date"].dt.year
df["sold_date_missing"] = df["sold_year"].isna().astype(int)

df = df.drop("prev_sold_date",axis = 1)
df

In [ ]:
false_year = df.loc[df["sold_year"] > 2026, "sold_year"]

df["sold_year"] = df["sold_year"].replace(3019, 2019)

In [ ]:
X = df.drop("price",axis = 1)
y = df["price"]

In [ ]:
Numerical_Features = X.select_dtypes('number').columns.to_list()
Numerical_Features

In [ ]:
Categorical_Features = X.select_dtypes(include=["object", "category", "string"]).columns.to_list()
Categorical_Features

In [ ]:
df.select_dtypes(include = "number").corr()

In [ ]:


sample = X[
    (X["house_size"] > 0) &
    (y > 0)
].sample(frac=0.1, random_state=42)

sns.scatterplot(
    data=sample,
    x="house_size",
    y=y,
    alpha=0.3
)

plt.xscale("log")
plt.yscale("log")
plt.title("House Size vs Price")
plt.show()

In [ ]:
Q1 = X["house_size"].quantile(0.25)
Q3 = X["house_size"].quantile(0.75)
IQR = Q3 - Q1
LowerBound = Q1 - 1.5*Q1
UpperBound = Q3 + 2.5*Q1
Outliers = X[X["house_size"]>UpperBound].nunique()
print("LowerBound: ", LowerBound)
print("UpperBound: ", UpperBound)
print("No. of Outliers: ", Outliers)

In [ ]:
data_outfree = df.drop(Outliers, axis = 0)
data_outfree.dropna(subset = "price", inplace = True)
data_outfree

In [ ]:
Xd = data_outfree.drop("price", axis = 1)
yd = data_outfree["price"]
print(Xd.shape)
print(yd.shape)

In [ ]:
Numerical_Features_d = Xd.select_dtypes(include= "number").columns.to_list()
Categorical_Features_d = Xd.select_dtypes(include = "object").columns.to_list()
print(Numerical_Features_d)
print(Categorical_Features_d)

In [ ]:
plt.figure(figsize = (15,8))
sns.heatmap(df.select_dtypes(include = "number").corr(), annot = True, cmap = "coolwarm", fmt = ".6f")
plt.show()

In [ ]:
plt.figure(figsize = (15,8))
sns.heatmap(data_outfree.select_dtypes(include = "number").corr(), annot = True, cmap = "coolwarm", fmt = ".6f")
plt.show()

In [ ]:
Numerical_Pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])


Categorical_Pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

Polynomial_Numerical_Pipeline = Pipeline([
    ("Imputer", SimpleImputer(strategy="median")),
    ("Polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("Scaler", StandardScaler())
])

In [ ]:
Transformer = ColumnTransformer([
    ("Numerical",Numerical_Pipeline, Numerical_Features),
    ("Categorical", Categorical_Pipeline, Categorical_Features)
])

Polynomial_Transformer = ColumnTransformer([
        ("Numerical",Polynomial_Numerical_Pipeline, Numerical_Features),
    ("Categorical", Categorical_Pipeline, Categorical_Features)
])



In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15/0.85, random_state=42
)
print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', X_test.shape)

In [ ]:
Linear_Model = Pipeline([
    ("Transformer", Transformer),
    ("Regressor", LinearRegression())
])

RandomForest_Model = Pipeline([
    ("Transformer", Transformer),
    ("Regressor", RandomForestRegressor())
])

Ridge_Model = Pipeline([
    ("Transformer", Transformer),
    ("Regressor", Ridge())
])

Lasso_Model = Pipeline([
    ("Transformer", Transformer),
    ("Regressor", Lasso())
])

ElasticNet_Model = Pipeline([
    ("Transformer", Transformer),
    ("Regressor", ElasticNet())
])

SGD_Model = Pipeline([
    ("Transformer", Transformer),
    ("Regressor", SGDRegressor())
])

GradientBoosting_Model = Pipeline([
    ("Transformer", clone(Transformer)),
    ("Regressor", GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

Polynomial_Model = Pipeline([
    ("Transformer", Polynomial_Transformer),
    ("Regressor", LinearRegression())
])



In [ ]:
results = []

def check_score(name, model, prediction):
    train_score = model.score(X_train, y_train)
    validation_score = r2_score(y_val, prediction)
    mae = mean_absolute_error(y_val, prediction)
    rmse = np.sqrt(mean_squared_error(y_val, prediction))
    results.append([name, train_score, validation_score, mae, rmse])
    print(name)
    print('Training R2:', train_score)
    print('Validation R2:', validation_score)
    print('MAE:', mae)
    print('RMSE:', rmse)


In [ ]:
Linear_Model.fit(X_train, y_train)
y_predict = Linear_Model.predict(X_val)
check_score('Linear Regression', Linear_Model, y_predict)


In [ ]:
Ridge_Model.fit(X_train, y_train)
y_predict = Ridge_Model.predict(X_val)
check_score('Ridge Regression', Ridge_Model, y_predict)


In [ ]:
Lasso_Model.fit(X_train, y_train)
y_predict = Lasso_Model.predict(X_val)
check_score('Lasso Regression', Lasso_Model, y_predict)


In [ ]:
ElasticNet_Model.fit(X_train, y_train)
y_predict = ElasticNet_Model.predict(X_val)
check_score('ElasticNet Regression', ElasticNet_Model, y_predict)


In [ ]:
SGD_Model.fit(X_train, y_train)
y_predict = SGD_Model.predict(X_val)
check_score('SGD Regression', SGD_Model, y_predict)

In [ ]:
Polynomial_Model.fit(X_train, y_train)
y_predict = Polynomial_Model.predict(X_val)
check_score('Polynomial Regression', Polynomial_Model, y_predict)


In [ ]:
RandomForest_Model.fit(X_train, y_train)
y_predict = RandomForest_Model.predict(X_val)
check_score('Random Forest', RandomForest_Model, y_predict)


In [ ]:
GradientBoosting_Model.fit(X_train, y_train)
y_predict = GradientBoosting_Model.predict(X_val)
check_score('Gradient Boosting', GradientBoosting_Model, y_predict)


In [ ]:
# XGBoost_Model.fit(X_train, y_train)
# XGBoost_Pred = XGBoost_Model.predict(X_val)
# print('XGBoost Training R2:', XGBoost_Model.score(X_train, y_train))
# print('XGBoost Validation R2:', r2_score(y_val, XGBoost_Pred))


### 10. Compare training vs validation scores

In [ ]:
student_scores = pd.DataFrame(results, columns=['Model','Train R2','Validation R2','MAE','RMSE'])
student_scores = student_scores.sort_values('Validation R2', ascending=False)
display(student_scores)
student_scores.to_csv('realtor_student_scores.csv', index=False)

student_scores.set_index('Model')[['Train R2','Validation R2']].plot(
    kind='bar', figsize=(11,5), rot=45
)
plt.title('Train vs validation R² (look for overfitting)')
plt.ylabel('R² score')
plt.tight_layout()
plt.show()
